In [11]:
# Cell 1 — imports and connection
import os
import pandas as pd
from dotenv import load_dotenv
import snowflake.connector
from collections import Counter
import plotly.express as px
import json

load_dotenv()

conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database='ANALYTICS_PROD',
    schema='PUBLIC',
)

def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)

print("Connected.")

Connected.


In [12]:
# Cell 2 — load mart
df = run_query("SELECT * FROM ANALYTICS_PROD.PUBLIC.FCT_JOB_POSTINGS")
df.columns = df.columns.str.lower()
print(f"Total rows: {len(df)}")
print(f"Enriched rows: {df['inferred_seniority'].notna().sum()}")
print(f"Columns: {list(df.columns)}")

def parse_variant(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return json.loads(val)
        except:
            return []
    return []

for col in ["tech_stack_required", "tech_stack_preferred", "paradigms_required", "paradigms_preferred"]:
    df[col] = df[col].apply(parse_variant)

Total rows: 210
Enriched rows: 208
Columns: ['job_id', 'source', 'ingestion_query', 'job_title', 'company_name', 'job_url', 'date_posted', 'description', 'city', 'state', 'country', 'latitude', 'longitude', 'work_model', 'employment_type', 'listed_seniority', 'is_explicitly_entry_level', 'final_salary_min', 'final_salary_max', 'inferred_seniority', 'role_archetype', 'work_focus', 'title_seniority_signal', 'title_signal_reasoning', 'acknowledges_ai', 'domain', 'explicitly_encourages_applicants', 'tech_stack_required', 'tech_stack_preferred', 'paradigms_required', 'paradigms_preferred', 'degree_requirement', 'years_required_min', 'years_required_max', 'confidence_score', 'enriched_at', 'ingested_at']


In [13]:
# Cell 3 — acknowledges_ai
print("=== acknowledges_ai ===")
print(df["acknowledges_ai"].value_counts())
print()
print("By ingestion_query:")
print(df.groupby("ingestion_query")["acknowledges_ai"].mean().sort_values(ascending=False))

=== acknowledges_ai ===
acknowledges_ai
False    141
True      67
Name: count, dtype: int64

By ingestion_query:
ingestion_query
Analytics Engineer    0.576923
Data Engineer         0.441176
Data Analyst              0.25
Name: acknowledges_ai, dtype: object


In [14]:
# Cell 4 — traditional vs modern stack salary gap
MODERN_SKILLS = {"dbt", "snowflake", "airflow", "spark", "kafka", "databricks"}
TRAD_SKILLS = {"excel", "tableau", "power bi", "sql server", "access", "sas", "stata"}

salary_df = df.dropna(subset=["final_salary_min", "final_salary_max"]).copy()
salary_df["salary_mid"] = (salary_df["final_salary_min"] + salary_df["final_salary_max"]) / 2

def classify_stack(skills):
    if not isinstance(skills, list):
        return "neither"
    skills_lower = {s.lower() for s in skills}
    if skills_lower & MODERN_SKILLS:
        return "modern"
    if skills_lower & TRAD_SKILLS:
        return "traditional"
    return "neither"

salary_df["stack_type"] = salary_df["tech_stack_required"].apply(classify_stack)
print("=== Stack salary gap ===")
print(salary_df.groupby("stack_type")["salary_mid"].agg(["median", "count"]))

=== Stack salary gap ===
               median  count
stack_type                  
modern       128400.0     25
neither      124500.0     49
traditional   87800.0     48


In [16]:
# Cell 5 — listed_seniority vs inferred_seniority
seniority_map = {
    "entry_level": "entry",
    "junior": "entry",
    "mid_level": "mid",
}

listed = df.dropna(subset=["listed_seniority", "inferred_seniority"]).copy()
listed["listed_normalized"] = listed["listed_seniority"].map(seniority_map)

print(f"=== listed vs inferred seniority (n={len(listed)}) ===")
print(pd.crosstab(listed["listed_normalized"], listed["inferred_seniority"]))
print()
agree = (listed["listed_normalized"] == listed["inferred_seniority"]).mean()
print(f"Agreement rate: {agree:.0%}")

=== listed vs inferred seniority (n=108) ===
inferred_seniority  entry  mid  senior
listed_normalized                     
entry                  14   12       0
mid                    15   56      11

Agreement rate: 65%


In [17]:
# Cell 6 — seniority ladder
print("=== Seniority ladder — listed_seniority ===")
ladder = df.dropna(subset=["listed_seniority"])
print(f"n={len(ladder)}")
print(ladder.groupby("listed_seniority")[["years_required_min", "final_salary_min"]].median())

=== Seniority ladder — listed_seniority ===
n=109
                  years_required_min  final_salary_min
listed_seniority                                      
entry_level                      NaN           81000.0
junior                           2.0           85000.0
mid_level                        3.0           96579.5


In [18]:
# Cell 7 — domain distribution
print("=== Domain fill rate ===")
print(f"Filled: {df['domain'].notna().sum()} / {len(df)} ({df['domain'].notna().mean():.0%})")
print()
print("=== Domain distribution ===")
print(df["domain"].value_counts())
print()
print("=== Role archetype by domain ===")
domain_df = df.dropna(subset=["domain", "role_archetype"])
print(pd.crosstab(domain_df["domain"], domain_df["role_archetype"]))

=== Domain fill rate ===
Filled: 180 / 210 (86%)

=== Domain distribution ===
domain
finance               38
healthcare            37
tech                  33
government            15
media                 10
retail                10
consulting            10
insurance              6
gaming                 4
education              4
energy                 2
nonprofit              2
transportation         2
financial services     1
fintech                1
beauty                 1
logistics              1
agriculture            1
entertainment          1
utilities              1
Name: count, dtype: int64

=== Role archetype by domain ===
role_archetype      analytics_engineer  data_analyst  data_engineer  hybrid
domain                                                                     
agriculture                          0             1              0       0
beauty                               1             0              0       0
consulting                           0             

In [19]:
# Cell 8 — explicitly_encourages_applicants
print("=== explicitly_encourages_applicants ===")
print(df["explicitly_encourages_applicants"].value_counts())
print()
print("By stack type (where salary available):")
print(salary_df.groupby("stack_type")["explicitly_encourages_applicants"].mean())

=== explicitly_encourages_applicants ===
explicitly_encourages_applicants
False    156
True      52
Name: count, dtype: int64

By stack type (where salary available):
stack_type
modern             0.32
neither        0.270833
traditional    0.395833
Name: explicitly_encourages_applicants, dtype: object


In [20]:
# Cell 9 — title_seniority_signal
print("=== title_seniority_signal ===")
print(df["title_seniority_signal"].value_counts())
print()
print("By ingestion_query:")
print(pd.crosstab(df["ingestion_query"], df["title_seniority_signal"], normalize="index").round(2))

=== title_seniority_signal ===
title_seniority_signal
accurate       185
overstated      14
understated      9
Name: count, dtype: int64

By ingestion_query:
title_seniority_signal  accurate  overstated  understated
ingestion_query                                          
Analytics Engineer          0.92        0.00         0.08
Data Analyst                0.87        0.08         0.05
Data Engineer               0.94        0.06         0.00


In [21]:
# Cell 10 — ingestion_query vs role_archetype divergence
query_to_archetype_map = {
    "Data Analyst": "data_analyst",
    "Analytics Engineer": "analytics_engineer",
    "Data Engineer": "data_engineer",
}

divergence_df = df.dropna(subset=["ingestion_query", "role_archetype"]).copy()
divergence_df["expected_archetype"] = divergence_df["ingestion_query"].map(query_to_archetype_map)
divergence_df["matches"] = divergence_df["role_archetype"] == divergence_df["expected_archetype"]

total = len(divergence_df)
matched = divergence_df["matches"].sum()
print(f"=== Overall agreement: {matched}/{total} ({matched/total:.0%}) ===\n")

print("=== Divergent combos (where LLM disagrees with query) ===")
divergent = divergence_df[~divergence_df["matches"]]
print(divergent.groupby(["ingestion_query", "role_archetype"]).size().sort_values(ascending=False))

=== Overall agreement: 176/208 (85%) ===

=== Divergent combos (where LLM disagrees with query) ===
ingestion_query     role_archetype
Analytics Engineer  data_engineer     15
Data Analyst        hybrid            12
                    data_engineer      2
Analytics Engineer  data_analyst       1
                    hybrid             1
Data Engineer       hybrid             1
dtype: int64
